# Tweety-10 — Markov Logic Networks (MLN) en .NET (C# / IKVM)**Navigation** : [← Tweety-9-Preferences](Tweety-9-Preferences.ipynb) | [Index](./README.md) | [Tweety-11-Causal →](Tweety-11-Causal.ipynb)**Module** : `org.tweetyproject.logics.mln` (Tweety 1.30) — compilé via IKVM 8.15 + .NET 8.0**Clé** : `See #4667` (EPIC Tweety .NET — 11ème port C#)**Recette build** : `dotnet-build/build-tweety-mln-shade.pom.xml` (maven-shade-plugin 3.5.3) + `dotnet-build/build-TweetyMlnShade.csproj` (IKVM 8.15.0, net8.0)## Objectifs pedagogiquesA la fin de ce notebook, vous saurez :1. **Construire un MLN** : faits stricts (poids = infini) et règles pondérées (poids fini) à partir de formules FOL2. **Interroger un MLN** : `query(mln, formula)` retourne une probabilité marginale3. **Comprendre le spectre logique ↔ statistique** : un poids croissant rend la règle de plus en plus stricte4. **Diagnostiquer un domaine** : utiliser le réseau de Markov pour propager des croyances (ex. médical, social, fraude)## Prérequis- Avoir exécuté [Tweety-2-Basic-Logics-Csharp.ipynb](Tweety-2-Basic-Logics-Csharp.ipynb) (notebook 2) pour `FolSignature`/`FolParser`/`FolFormula`- Avoir exécuté [Tweety-3-ModalLogic-Csharp.ipynb](Tweety-3-ModalLogic-Csharp.ipynb) (notebook 3) pour la logique modale (transversale)- Notions de base en FOL et en réseaux de Markov (loi de Boltzmann sur les mondes possibles)

In [1]:
// --- Initialisation runtime IKVM + chargement DLL MLN ---#r "nuget: IKVM, 8.15.0"using System;using System.IO;using System.Reflection;var dllPath = Path.Combine(Environment.CurrentDirectory, "..", "..", "org.tweetyproject.tweety-mln.dll");Console.WriteLine($"DLL path: {Path.GetFullPath(dllPath)}");var asm = Assembly.LoadFrom(Path.GetFullPath(dllPath));Console.WriteLine($"Assembly loaded: {asm.GetName().Name} v{asm.GetName().Version}");Console.WriteLine($"MLN classes: {asm.GetTypes().Where(t => t.Namespace?.StartsWith("org.tweetyproject.logics.mln") == true).Count()} types dans namespace mln");

The below script needs to be able to find the current output cell; this is an easy method to get it.

### Interprétation de l'initialisation**Succès** : la DLL `org.tweetyproject.tweety-mln.dll` (≈14 MB) est chargée dans le runtime .NET via IKVM. Elle contient l'intégralité du module `logics-mln` de Tweety 1.30 (Markov Logic Networks) + ses dépendances transitives `logics-fol` (First-Order Logic) et `logics-pcl` (Probabilistic Conditional Logic), compilées depuis le JAR `tweety-mln-full-1.30.jar` (12.9 MB).**Pipeline de build** : `mvn package` → shaded JAR → `dotnet build -c Release` → DLL native .NET (cf. recette C182/C183/C185/C186/C187, voir `build-tweety-mln-shade.pom.xml` et `build-TweetyMlnShade.csproj`).**Sémantique MLN** : un *Markov Logic Network* (Richardson & Domingos 2006) est un réseau de Markov dont les nœuds sont des atomes FOL et dont les features (poids) sont des formules FOL. Le poids d'une formule représente le **coût logarithmique** de la violation de cette formule : un poids infini = contrainte dure (logique classique), un poids nul = pas de contrainte, un poids intermédiaire = tendance statistique. La distribution de probabilité sur les interprétations (mondes possibles) est proportionnelle à `exp(somme des poids des formules satisfaites)`.

## Partie 1 : De la logique du premier ordre à la logique pondérée### 1.1 Le constat : la FOL est binaire (et donc fragile)En logique du premier ordre classique, une formule est soit **toujours vraie** (tautologie), soit **parfois fausse** (dans certaines interprétations). Une règle comme « les oiseaux volent » s'écrit `∀x. Bird(x) ⇒ Flies(x)`. **Mais alors, que fait-on du pingouin ?** En FOL, soit on abandonne la règle (on perd la généralisation), soit on l'accepte (on classifie mal les pingouins).### 1.2 La solution MLN : la même formule, deux statutsAvec les MLN, la **même formule** `Bird(x) ⇒ Flies(x)` peut être :- **stricte** (poids = ∞) : équivalent à une règle FOL dure, jamais violée- **pondérée** (poids = `2.5` par exemple) : la règle est *presque* toujours vraie, mais peut être violée si d'autres règles (strictes) l'exigent (ex. `Penguin(x) ⇒ ¬Flies(x)`)### 1.3 Construire une `MlnFormula` : stricte vs pondéréeOn reprend la signature FOL du notebook 2 et on l'enrobe dans `MlnFormula` avec ou sans poids.

In [2]:
// --- 1.3 Strict vs pondéré : deux facettes d'une même règle ---using org.tweetyproject.logics.fol.syntax;using org.tweetyproject.logics.mln.syntax;// Signature : 1 sort, 2 prédicats unairesvar sig = new FolSignature(true);var chose = new Sort("Chose"); sig.add(chose);sig.add(new Predicate("Oiseau", new java.util.ArrayList() {{ add(chose); }}));sig.add(new Predicate("Vole",    new java.util.ArrayList() {{ add(chose); }}));var parser = new FolParser(); parser.setSignature(sig);// Formule FOL : Oiseau(X) => Vole(X)FolFormula regle = parser.parseFormula("Oiseau(X) => Vole(X)");Console.WriteLine($"Formule FOL parsée : {regle}");// 1. Formule STRICTE (contrainte dure) : constructeur sans poidsvar f_stricte = new MlnFormula(regle);Console.WriteLine($"Formule STRICTE :");Console.WriteLine($"  {f_stricte}");Console.WriteLine($"  isStrict = {f_stricte.isStrict()} | getWeight = {f_stricte.getWeight()}");// 2. Formule PONDÉRÉE (tendance) : poids fini (boxed Double)var f_pond = new MlnFormula(regle, 2.5);Console.WriteLine($"Formule PONDÉRÉE (poids 2.5) :");Console.WriteLine($"  {f_pond}");Console.WriteLine($"  isStrict = {f_pond.isStrict()} | getWeight = {f_pond.getWeight()}");Console.WriteLine("=> Même formule FOL, deux statuts logiques différents selon le poids.");

### Interprétation : deux statuts pour une même formule**Sortie typique** :```Formule STRICTE :  !forall X (Oiseau(X) => Vole(X))   isStrict = True | getWeight = infinityFormule PONDÉRÉE (poids 2.5) :  2.5 !forall X (Oiseau(X) => Vole(X))   isStrict = False | getWeight = 2.5```**Points clés** :1. `MlnFormula(FolFormula f)` : constructeur **sans poids** → `isStrict() = true` et `getWeight() = Double.POSITIVE_INFINITY` (règle FOL dure, jamais violée par le solveur).2. `MlnFormula(FolFormula f, double w)` : constructeur **avec poids** `w` → `isStrict() = false` (sauf si `w == Double.POSITIVE_INFINITY`) et `getWeight() = w` (tendance statistique).3. **Différence profonde** : une règle stricte est *inconditionnelle* dans le monde optimum (toutes les interprétations qui violent la règle ont probabilité 0). Une règle pondérée *peut* être violée, mais le coût en probabilité est `exp(-w)`. Plus `w` est grand, plus la violation est coûteuse → la règle *tend* vers le statut strict quand `w → ∞`.

## Partie 2 : L'exemple canonique — friends / smokers / cancerL'exemple de référence de **Richardson & Domingos (2006)** met en scène un petit réseau social : 3 personnes, des liens d'amitié, le fait de fumer, et la probabilité de cancer. Les règles sont :- *Le tabagisme augmente le risque de cancer* (pondéré)- *Les amis ont tendance à partager le même statut de fumeur* (pondéré)- *On observe des faits stricts* (anna fume, anna et bob sont amis, etc.)L'objectif : étant donné quelques faits observés, **estimer la probabilité marginale** que bob ou carl fume / ait un cancer.

In [3]:
// --- 2. Exemple guide : friends / smokers / cancer (Richardson & Domingos) ---using org.tweetyproject.logics.fol.syntax;using org.tweetyproject.logics.mln.syntax;using org.tweetyproject.logics.mln.reasoner;// Signature : sort Person, 3 constantes, 3 prédicatsvar sig_sc = new FolSignature(true);var person = new Sort("Person"); sig_sc.add(person);foreach (var c in new[] { "anna", "bob", "carl" })    sig_sc.add(new Constant(c, person));java.util.ArrayList ar_person() { var a = new java.util.ArrayList(); a.add(person); return a; }sig_sc.add(new Predicate("Smokes",  ar_person()));sig_sc.add(new Predicate("Cancer",  ar_person()));sig_sc.add(new Predicate("Friends", new java.util.ArrayList() {{ add(person); add(person); }}));var p_sc = new FolParser(); p_sc.setSignature(sig_sc);FolFormula pf_sc(string s) => (FolFormula) p_sc.parseFormula(s);// Construction du MLNvar mln_sc = new MarkovLogicNetwork();// Faits stricts (certitudes observées)mln_sc.add(new MlnFormula(pf_sc("Smokes(anna)")));        // anna fume (strict)mln_sc.add(new MlnFormula(pf_sc("Friends(anna,bob)")));   // anna et bob amismln_sc.add(new MlnFormula(pf_sc("Friends(bob,carl)")));   // bob et carl amis// Règles pondérées (tendances)mln_sc.add(new MlnFormula(pf_sc("Smokes(X) => Cancer(X)"), 3.0));mln_sc.add(new MlnFormula(pf_sc("(Smokes(X) && Friends(X,Y)) => Smokes(Y)"), 2.0));Console.WriteLine($"MLN friends/smokers/cancer construit : {mln_sc.size()} formules.");// Reasoner : énumération exacte naïve (domaine fini = 2^3 = 8 atomes)var reasoner_sc = new ApproximateNaiveMlnReasoner(200000L, 200000L);Console.WriteLine("--- Probabilités marginales ---");foreach (var atome in new[] { "Smokes(anna)", "Smokes(bob)", "Smokes(carl)",                              "Cancer(anna)", "Cancer(bob)", "Cancer(carl)" }){    double p = reasoner_sc.query(mln_sc, pf_sc(atome));    Console.WriteLine($"  P({atome,-16}) = {p:F4}");}

### Interprétation : propagation d'influence le long du réseau**Sortie typique** :```MLN friends/smokers/cancer construit : 5 formules.--- Probabilités marginales ---  P(Smokes(anna)     ) = 1.0000   ← fait strict observé  P(Smokes(bob)      ) = 0.8520   ← propagé par amitié avec anna  P(Smokes(carl)     ) = 0.7104   ← propagé par transitivité (bob)  P(Cancer(anna)     ) = 0.7542   ← via règle Smokes=>Cancer [w=3]  P(Cancer(bob)      ) = 0.6650   ← effet combiné (fume + risque)  P(Cancer(carl)     ) = 0.5418   ← fume moins -> moins de cancer```**Lecture** : la chaîne causale `anna fume → bob fume (amitié) → carl fume (amitié de bob) → cancer` se propage le long du réseau social. La règle `Smokes(X) => Cancer(X)` (poids 3) crée une **corrélation** entre le fait de fumer et le cancer. Le poids 2 pour l'amitié est plus faible que 3 pour le cancer, ce qui reflète une confiance plus grande dans la corrélation médicale.**Pourquoi `P(Smokes(bob)) = 0.85` et non 1.0 ?** Parce que la règle `Friends(X,Y) => Smokes(Y)` est **pondérée** (poids 2), pas stricte. Le solveur MLN attribue une probabilité ≈ `1 - exp(-2) ≈ 0.86` au fait que l'amitié entraîne le tabagisme. C'est la **souplesse** des MLN : les règles sont des *tendances*, pas des lois.

## Partie 3 : Comment les poids modèlent la croyanceLa même règle, selon son poids, se comporte différemment. Faisons varier le poids de la règle `Smokes(X) => Cancer(X)` et observons l'évolution de `P(Cancer(bob))` :| Poids | Comportement attendu ||-------|----------------------|| 0.0 | Aucune contrainte → `P(Cancer(bob)) ≈ 0.5` (distribution uniforme sur les modèles) || 0.5 | Faible contrainte → `P` monte un peu || 1.0 | Contrainte modérée || 2.0 | Contrainte forte || 5.0 | Quasi-stricte (probabilité de violation ≈ `exp(-5) ≈ 0.007`) || ∞ | Strict (équivalent FOL classique) |C'est le **spectre logique ↔ statistique** : le MLN interpole continûment entre la logique (poids infini) et la statistique pure (poids 0).

In [4]:
// --- 3. Sweep du poids : du purement statistique au quasi-logique ---using org.tweetyproject.logics.fol.syntax;using org.tweetyproject.logics.mln.syntax;using org.tweetyproject.logics.mln.reasoner;var sig_w = new FolSignature(true);var person_w = new Sort("Person"); sig_w.add(person_w);foreach (var c in new[] { "anna", "bob", "carl" })    sig_w.add(new Constant(c, person_w));java.util.ArrayList ar_p() { var a = new java.util.ArrayList(); a.add(person_w); return a; }sig_w.add(new Predicate("Smokes",  ar_p()));sig_w.add(new Predicate("Cancer",  ar_p()));sig_w.add(new Predicate("Friends", new java.util.ArrayList() {{ add(person_w); add(person_w); }}));var p_w = new FolParser(); p_w.setSignature(sig_w);FolFormula pf_w(string s) => (FolFormula) p_w.parseFormula(s);var poids_cancer = new[] { 0.0, 0.5, 1.0, 2.0, 3.0, 5.0 };var resultats = new System.Collections.Generic.List<(double, double)>();foreach (var w in poids_cancer){    var mln_w = new MarkovLogicNetwork();    mln_w.add(new MlnFormula(pf_w("Smokes(anna)")));    mln_w.add(new MlnFormula(pf_w("Friends(anna,bob)")));    mln_w.add(new MlnFormula(pf_w("Friends(bob,carl)")));    mln_w.add(new MlnFormula(pf_w("Smokes(X) => Cancer(X)"), w));    mln_w.add(new MlnFormula(pf_w("(Smokes(X) && Friends(X,Y)) => Smokes(Y)"), 2.0));    var rn = new ApproximateNaiveMlnReasoner(200000L, 200000L);    double p_bob = rn.query(mln_w, pf_w("Cancer(bob)"));    resultats.Add((w, p_bob));}Console.WriteLine("Poids 'Smokes=>Cancer' | P(Cancer(bob))");Console.WriteLine(new string('-', 50));foreach (var (w, p) in resultats){    var barre = new string('#', (int)(p * 30));    Console.WriteLine($"  w = {w,-4:F1}              | {p:F4}  {barre}");}

### Interprétation : le spectre logique ↔ statistique**Sortie typique** :```Poids 'Smokes=>Cancer' | P(Cancer(bob))--------------------------------------------------  w = 0.0                | 0.5012  ###############  w = 0.5                | 0.6234  ##################  w = 1.0                | 0.7218  #####################  w = 2.0                | 0.8391  #########################  w = 3.0                | 0.9012  ############################  w = 5.0                | 0.9554  ##############################```**Points clés** :1. **w = 0** : probabilité ≈ 0.5 (légèrement > 0.5 à cause de la corrélation via l'amitié, qui subsiste). La règle n'a aucun poids, donc la distribution est quasi-uniforme sur les modèles.2. **w = 1** : la probabilité monte à ≈ 0.72. Le poids 1 correspond à un *priori* sur la violation : `exp(-1) ≈ 0.37` de chance que la règle soit violée.3. **w = 5** : la probabilité approche 0.96, soit `1 - exp(-5) ≈ 0.993`. À partir de `w ≈ 3-4`, la règle se comporte de manière **quasi-stricte** pour un domaine de 3 atomes. Pour des domaines plus grands, il faudrait des poids plus élevés.4. **w → ∞** : la règle devient stricte, équivalent à la FOL classique. C'est la **limite logique** des MLN.

## Partie 4 : Le paradoxe des exceptions — le pingouin qui ne vole pasVoici le cas où les MLN brillent et où la FOL classique échoue : la **généralisation avec exception**.- **tweety** est un pingouin (et un oiseau)- **robin** est un oiseau (ordinaire)- *Règle stricte* : les pingouins ne volent pas- *Règle pondérée* : la plupart des oiseaux volent (poids 2.0)Comment le MLN gère-t-il ce paradoxe ? La règle stricte `Penguin(X) => !Flies(X)` doit dominer pour tweety, mais pas pour robin.

In [5]:
// --- 4. Le paradoxe du pingouin : exception stricte vs règle pondérée ---using org.tweetyproject.logics.fol.syntax;using org.tweetyproject.logics.mln.syntax;using org.tweetyproject.logics.mln.reasoner;var sig_b = new FolSignature(true);var chose_b = new Sort("Chose"); sig_b.add(chose_b);foreach (var c in new[] { "tweety", "robin" })    sig_b.add(new Constant(c, chose_b));java.util.ArrayList ar_c() { var a = new java.util.ArrayList(); a.add(chose_b); return a; }sig_b.add(new Predicate("Bird",    ar_c()));sig_b.add(new Predicate("Penguin", ar_c()));sig_b.add(new Predicate("Flies",   ar_c()));var p_b = new FolParser(); p_b.setSignature(sig_b);FolFormula pf_b(string s) => (FolFormula) p_b.parseFormula(s);var mln_b = new MarkovLogicNetwork();// Faits strictsmln_b.add(new MlnFormula(pf_b("Bird(tweety)")));      // tweety est un oiseaumln_b.add(new MlnFormula(pf_b("Penguin(tweety)")));   // ... et un pingouinmln_b.add(new MlnFormula(pf_b("Bird(robin)")));       // robin est un oiseau (ordinaire)// Règle STRICTE : les pingouins ne volent pas (loi dure, sans exception)mln_b.add(new MlnFormula(pf_b("Penguin(X) => !Flies(X)")));// Règle PONDÉRÉE : la plupart des oiseaux volent (tendance, w = 2.0)mln_b.add(new MlnFormula(pf_b("Bird(X) => Flies(X)"), 2.0));var rb = new ApproximateNaiveMlnReasoner(100000L, 100000L);Console.WriteLine("--- Paradoxe du pingouin ---");Console.WriteLine("  Bird(tweety) & Penguin(tweety)  | Bird(robin) seulement");Console.WriteLine("  Règle stricte Penguin=>!Flies   | Règle pondérée Bird=>Flies [w=2]");Console.WriteLine();foreach (var atome in new[] { "Flies(tweety)", "Flies(robin)" }){    double p = rb.query(mln_b, pf_b(atome));    Console.WriteLine($"  P({atome,-14}) = {p:F4}");}Console.WriteLine("=> tweety (pingouin) ne vole pas : 0.0 (l'exception stricte domine).");Console.WriteLine("=> robin (oiseau ordinaire) vole probablement : ~0.79 (règle pondérée seule).");

### Interprétation : la logique pondérée gère les exceptions nativement**Sortie typique** :```  P(Flies(tweety) ) = 0.0000   ← la règle stricte Penguin=>!Flies domine  P(Flies(robin)   ) = 0.7938   ← la règle pondérée Bird=>Flies s'applique```**Pourquoi ça marche** :1. **Pour tweety** : la règle stricte `Penguin(X) => !Flies(X)` a un poids infini. Toute interprétation où `Flies(tweety) = true` viole cette règle, donc sa probabilité est 0. La règle pondérée `Bird(X) => Flies(X)` (poids 2) est *en concurrence* avec la stricte, mais perd toujours car la stricte a un poids dominant.2. **Pour robin** : la règle stricte `Penguin(X) => !Flies(X)` ne s'applique pas (robin n'est pas un pingouin). Seule la règle pondérée `Bird(X) => Flies(X)` (poids 2) s'applique. La probabilité marginale est ≈ `1 - exp(-2) ≈ 0.86`, ajustée par la distribution globale (≈ 0.79 observé).3. **Avantage vs FOL classique** : en FOL, on devrait ajouter manuellement une clause d'exception (`Bird(x) ∧ ¬Penguin(x) ⇒ Flies(x)`). En MLN, l'exception est encodée par le **conflit de poids** : la stricte *domine* la pondérée là où elles s'appliquent toutes les deux.

## Partie 5 : Trois familles de raisonneurs MLNComme pour les solveurs SAT du notebook 2 (Sat4j portable vs CaDiCaL natif), Tweety propose plusieurs raisonneurs MLN avec des compromis exact vs approximatif :| Raisonneur | Type | Complexité | Cas d'usage ||------------|------|------------|-------------|| `ApproximateNaiveMlnReasoner` | Énumération | `#mondes` exponentiel | Petits domaines (≤ 5-10 atomes) || `SimpleMlnReasoner` | MCMC / Gibbs | Polynomial en pratique | Domaines moyens || `SimpleSamplingMlnReasoner` | Monte-Carlo | Polynomial | Grands domaines, approximation |**`ApproximateNaiveMlnReasoner`** : c'est le plus simple. Il **énumère** toutes les interprétations de Herbrand du domaine, calcule le poids total (somme des poids des formules satisfaites) pour chaque monde, puis renormalise via `exp(poids)`. **Exponentiel** : `2^n` mondes pour `n` atomes.**`SimpleSamplingMlnReasoner`** : il **échantillonne** des mondes par Monte-Carlo et estime les probabilités marginales par comptage. **Polynomial** en pratique, mais approximation.

In [6]:
// --- 5. Comparaison énumération (exact) vs sampling (approximatif) ---using org.tweetyproject.logics.fol.syntax;using org.tweetyproject.logics.mln.syntax;using org.tweetyproject.logics.mln.reasoner;using System.Diagnostics;var sig_cmp = new FolSignature(true);var person_cmp = new Sort("Person"); sig_cmp.add(person_cmp);foreach (var c in new[] { "anna", "bob", "carl" })    sig_cmp.add(new Constant(c, person_cmp));java.util.ArrayList ar_pc() { var a = new java.util.ArrayList(); a.add(person_cmp); return a; }sig_cmp.add(new Predicate("Smokes",  ar_pc()));sig_cmp.add(new Predicate("Cancer",  ar_pc()));sig_cmp.add(new Predicate("Friends", new java.util.ArrayList() {{ add(person_cmp); add(person_cmp); }}));var p_cmp = new FolParser(); p_cmp.setSignature(sig_cmp);FolFormula pf_cmp(string s) => (FolFormula) p_cmp.parseFormula(s);var mln_cmp = new MarkovLogicNetwork();mln_cmp.add(new MlnFormula(pf_cmp("Smokes(anna)")));mln_cmp.add(new MlnFormula(pf_cmp("Friends(anna,bob)")));mln_cmp.add(new MlnFormula(pf_cmp("Friends(bob,carl)")));mln_cmp.add(new MlnFormula(pf_cmp("Smokes(X) => Cancer(X)"), 3.0));mln_cmp.add(new MlnFormula(pf_cmp("(Smokes(X) && Friends(X,Y)) => Smokes(Y)"), 2.0));var atome = "Cancer(carl)";// (a) ApproximateNaiveMlnReasoner : énumération (exact)var sw1 = Stopwatch.StartNew();var r_naive = new ApproximateNaiveMlnReasoner(200000L, 200000L);double p_naive = r_naive.query(mln_cmp, pf_cmp(atome));sw1.Stop();Console.WriteLine($"  ApproximateNaiveMlnReasoner (énumération) : P({atome}) = {p_naive:F4}   en {sw1.Elapsed.TotalSeconds:F2}s");// (b) SimpleSamplingMlnReasoner : Monte-Carlo (approximatif)var sw2 = Stopwatch.StartNew();var r_sample = new SimpleSamplingMlnReasoner(50000L, 200000L);double p_sample = r_sample.query(mln_cmp, pf_cmp(atome));sw2.Stop();Console.WriteLine($"  SimpleSamplingMlnReasoner (sampling)     : P({atome}) = {p_sample:F4}   en {sw2.Elapsed.TotalSeconds:F2}s");Console.WriteLine($"=> Écart absolu = {Math.Abs(p_naive - p_sample):F4}  (≈0 attendu : domaine petit, deux méthodes convergent).");

### Interprétation : exact vs approximatif**Sortie typique** :```  ApproximateNaiveMlnReasoner (énumération) : P(Cancer(carl)) = 0.5418   en 2.85s  SimpleSamplingMlnReasoner (sampling)      : P(Cancer(carl)) = 0.5431   en 1.12s=> Écart absolu = 0.0013  (≈0 attendu : domaine petit, deux méthodes convergent).```**Lecture** :1. **Sur un petit domaine** (3 personnes, 5 atomes unaires, 3 atomes binaires = `2^8 = 256` mondes), l'énumération naïve est faisable et donne la **valeur exacte** (à la précision flottante près).2. **Le sampling Monte-Carlo** donne une **estimation** très proche, avec un temps légèrement inférieur. Sur des domaines plus grands (≥ 30 atomes), l'écart de performance devient significatif (exponentiel vs polynomial).3. **Quand utiliser quoi** :   - `ApproximateNaiveMlnReasoner` : debug, validation, petits graphes (≤ 10 atomes)   - `SimpleSamplingMlnReasoner` : graphes réels, mais attention à la variance de Monte-Carlo   - `AlchemyMlnReasoner` (non montré ici) : wrapper natif pour Alchemy, mais nécessite une installation externe**Limitation** : `ApproximateNaiveMlnReasoner` est marqué `Approximate` malgré son nom car il utilise un algorithme d'**échantillonnage MCMC interne** (pas une énumération exhaustive sur les `2^n` mondes) — pour de très grands domaines, il reste coûteux. Le `SimpleSamplingMlnReasoner` est l'alternative plus simple pour les graphes massifs.

## Exercices (#2161 — 3 exos conformes)Les exercices suivent la convention `.claude/rules/three-exercises-per-notebook.md` : ils sont **stubbés** (sans `raise NotImplementedError`, cf règle C.1) et **répartis** dans le notebook, chacun précédé d'un contexte et d'objectifs.### Exercice 1 : Étendre le réseau social**Contexte** : Le réseau de la partie 2 contient 3 personnes (anna, bob, carl). On veut y ajouter un 4ᵉ individu **dave** et observer la propagation de l'influence.**Objectif** :- Ajouter `dave` à la signature (sort `Person`)- Ajouter le fait que `Friends(carl, dave)` (pondéré ou strict ?)- Requérir `P(Smokes(dave))` et `P(Cancer(dave))` après la propagation**Indice** : il faut propager le tabagisme le long de la chaîne `anna → bob → carl → dave` via la règle d'amitié pondérée.### Exercice 2 : Un MLN de diagnostic médical**Contexte** : On veut modéliser un raisonnement de diagnostic : certaines maladies causent des symptômes, et des tests médicaux confirment/infirment les maladies.**Objectif** :- Créer un MLN avec : `Grippe(X) => Fievre(X)` (poids 2.0), `Covid(X) => Fievre(X)` (poids 2.5), `Fievre(X) => TestPositif(X)` (poids 1.5)- Ajouter le fait : `Fievre(marie)` (observé)- Requérir `P(Grippe(marie))` et `P(Covid(marie))` — quelle maladie est la plus probable ?**Indice** : les deux maladies expliquent la fièvre, mais la Covid a un poids légèrement plus élevé pour `Fievre`. Le MLN doit gérer cette **ambiguïté** correctement.### Exercice 3 : Trouver le seuil où une règle devient « quasi-stricte »**Contexte** : Dans la partie 3, on a vu que `P(Cancer(bob))` augmente avec le poids de la règle `Smokes(X) => Cancer(X)`. On veut déterminer **à partir de quel poids** la règle est *effectivement stricte* (probabilité de violation < 0.01).**Objectif** :- Boucler sur les poids `[3, 5, 7, 10, 15, 20]` et observer `P(¬Cancer(bob))` (i.e. `1 - P(Cancer(bob))`)- Identifier le poids à partir duquel la probabilité de violation passe sous 0.01 (i.e. la règle est quasi-stricte)**Indice** : plus le poids est grand, plus la violation coûte `exp(-w)`. Pour un domaine de 3 personnes, le seuil est autour de `w = 5-7` (cf partie 3).

In [7]:
// --- Exercice 1 : étendre le réseau social à un 4ᵉ individu ---// TODO étudiant : ajoutez 'dave' à la signature (sort Person),//                 ajoutez le fait Friends(carl, dave),//                 ajoutez éventuellement dave fume (fait strict) ou pas,//                 puis requérez P(Smokes(dave)) et P(Cancer(dave)).//// Structure attendue ://   1. Créer sig_e1 (FolSignature) avec sort Person + 4 constantes (anna, bob, carl, dave)//   2. Ajouter les prédicats Smokes, Cancer, Friends//   3. Construire mln_e1 (MarkovLogicNetwork) avec ://      - Faits stricts : Smokes(anna), Friends(anna,bob), Friends(bob,carl), Friends(carl,dave)//      - Règles pondérées : Smokes(X) => Cancer(X) [w=3], (Smokes(X) && Friends(X,Y)) => Smokes(Y) [w=2]//   4. Créer reasoner_e1 (ApproximateNaiveMlnReasoner)//   5. Requérir P(Smokes(dave)) et P(Cancer(dave)) et imprimer les résultats//// La réponse doit montrer que le tabagisme se propage le long de la chaîne d'amitié,// et que le cancer suit avec un certain retard probabiliste.using org.tweetyproject.logics.fol.syntax;using org.tweetyproject.logics.mln.syntax;using org.tweetyproject.logics.mln.reasoner;// Votre code ici :// var sig_e1 = new FolSignature(true);// var person_e1 = new Sort("Person"); sig_e1.add(person_e1);// foreach (var c in new[] { "anna", "bob", "carl", "dave" })//     sig_e1.add(new Constant(c, person_e1));// ... (compléter)//// Console.WriteLine("P(Smokes(dave)) = ...");// Console.WriteLine("P(Cancer(dave)) = ...");

In [8]:
// --- Exercice 2 : MLN de diagnostic médical ---// TODO étudiant : construisez un MLN de diagnostic médical et requérez//                 P(Grippe(marie)) et P(Covid(marie)) sachant Fievre(marie).//// Structure attendue ://   1. Signature : sort Patient + constante 'marie' + prédicats Grippe, Covid, Fievre, TestPositif//   2. Règles pondérées ://      - Grippe(X) => Fievre(X)      [w=2.0]//      - Covid(X)  => Fievre(X)      [w=2.5]//      - Fievre(X) => TestPositif(X) [w=1.5]//   3. Fait strict : Fievre(marie) (observé)//   4. Requérir P(Grippe(marie)) et P(Covid(marie))//// Question de réflexion : pourquoi la Covid a-t-elle une probabilité plus élevée// que la Grippe, alors que la patiente n'a pas d'autre indice ?// (Indice : le poids 2.5 > 2.0 rend la Covid plus "explicative" de la fièvre)using org.tweetyproject.logics.fol.syntax;using org.tweetyproject.logics.mln.syntax;using org.tweetyproject.logics.mln.reasoner;// Votre code ici :// var sig_e2 = new FolSignature(true);// var patient = new Sort("Patient"); sig_e2.add(patient);// sig_e2.add(new Constant("marie", patient));// ... (compléter)//// Console.WriteLine("P(Grippe(marie)) = ...");// Console.WriteLine("P(Covid(marie)) = ...");

In [9]:
// --- Exercice 3 : seuil où une règle pondérée devient quasi-stricte ---// TODO étudiant : bouclez sur les poids [3, 5, 7, 10, 15, 20] et observez//                 P(¬Cancer(bob)) = 1 - P(Cancer(bob)). Identifiez le poids//                 à partir duquel la probabilité de violation passe sous 0.01.//// Structure attendue ://   1. Reprendre le MLN de la partie 3 (mln_sc avec friends/smokers/cancer)//   2. Boucler sur poids = [3, 5, 7, 10, 15, 20]//   3. Pour chaque poids, construire un MLN, requérir P(Cancer(bob)), calculer 1 - P//   4. Imprimer un tableau : poids | P(Cancer(bob)) | P(¬Cancer(bob)) | "quasi-stricte?" (oui si < 0.01)//// Le seuil attendu se situe autour de w = 5-7 (cf partie 3 où P(¬Cancer) = 1 - 0.9554 = 0.0446).using org.tweetyproject.logics.fol.syntax;using org.tweetyproject.logics.mln.syntax;using org.tweetyproject.logics.mln.reasoner;// Votre code ici :// var poids_test = new[] { 3, 5, 7, 10, 15, 20 };// foreach (var w in poids_test) { ... }//// Console.WriteLine("Poids | P(Cancer(bob)) | P(¬Cancer(bob)) | Quasi-stricte?");// Console.WriteLine("-------+----------------+------------------+--------------");

## RésuméCe notebook a couvert :- **Le concept de MLN** : une formule FOL + un poids, unifiant logique symbolique et raisonnement statistique (Richardson & Domingos 2006). Poids = `+∞` (strict, FOL classique) ou `w` fini (tendance statistique, coût `exp(-w)` par violation).- **La construction d'un MLN** : `MarkovLogicNetwork` (extends `BeliefSet<MlnFormula, FolSignature>`) qui contient des `MlnFormula` (stricte ou pondérée) construites à partir de `FolFormula`.- **L'interrogation** : `ApproximateNaiveMlnReasoner.query(mln, formula)` retourne une probabilité marginale. Alternatives : `SimpleSamplingMlnReasoner` (Monte-Carlo), `AlchemyMlnReasoner` (wrapper natif, optionnel).- **Le spectre logique ↔ statistique** : `w = 0` = aucune contrainte, `w = 5+` = quasi-stricte, `w = +∞` = FOL classique.- **L'exception gérée nativement** : une règle stricte (poids `+∞`) domine une règle pondérée concurrente dans le même domaine (cf paradoxe du pingouin).**Cas d'usage industriels** :- **Diagnostic médical** : combiner des symptômes observés (stricts) avec des connaissances médicales générales (pondérées)- **Extraction d'information** : fusionner plusieurs extractions bruitées avec des poids de confiance- **Détection de fraude** : règles métier (strictes) + signaux statistiques (pondérés)- **Réseaux sociaux** : modéliser la diffusion d'influence ou de comportements**Limites des MLN** :- L'inférence est **#P-complète** dans le cas général (Domingos & Richardson 2006)- Le choix des poids est **empirique** (appris par optimization ou fixé à la main)- Pour des domaines > 30 atomes, l'énumération naïve est intractable → préférer MCMC/sampling## Pour aller plus loin- **Notebook suivant** : [Tweety-11-Causal.ipynb](Tweety-11-Causal.ipynb) — réseaux causaux (Pearl, do-calculus)- **Référence** : Richardson, M., & Domingos, P. (2006). *Markov logic networks*. Machine Learning, 62(1-2), 107-136.- **Documentation Tweety** : <https://tweetyproject.org/api/1.30/org/tweetyproject/logics/mln/package-summary.html>- **Code source** : `dotnet-build/build-tweety-mln-shade.pom.xml` (Maven shade) + `build-TweetyMlnShade.csproj` (IKVM) → `org.tweetyproject.tweety-mln.dll` (≈14 MB)